In [ ]:
import os
import re
import rasterio
import numpy as np
from rasterio.mask import mask
from shapely.ops import unary_union
import seaborn as sns
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss


## Balanging the final merged dataset

In [4]:
merged_df = pd.read_csv("Cleaned_dataset/grid/MERGED_FINAL.csv")

num_duplicates = merged_df.duplicated().sum()

print(f'The number of duplicated rows is: {num_duplicates}')
print(merged_df.shape)

merged_df = merged_df.drop(columns=["Unnamed: 0.1", "Unnamed: 0", 'HWSD2_SMU_ID'], errors='ignore')

num_duplicates = merged_df.duplicated().sum()

print(f'The number of duplicated rows is: {num_duplicates}')
print(merged_df.shape)
print(merged_df.columns)



The number of duplicated rows is: 117768
(851953, 40)
The number of duplicated rows is: 117768
(851953, 40)
Index(['spring_prec', 'autumn_prec', 'summer_prec', 'winter_prec',
       'autumn_tmax', 'elevation', 'autumn_tmin', 'spring_tmax', 'summer_tmax',
       'winter_tmin', 'spring_tmin', 'winter_tmax', 'summer_tmin', 'TOTAL_N',
       'ORG_CARBON', 'PH_WATER', 'CEC_EFF', 'TCARBON_EQ', 'SILT',
       'GRIDCODE_130.0', 'SAND', 'CN_RATIO', 'GRIDCODE_30.0', 'REF_BULK',
       'GRIDCODE_20.0', 'CEC_CLAY', 'ESP', 'COARSE', 'BULK', 'GRIDCODE_14.0',
       'CEC_SOIL', 'GYPSUM', 'BSAT', 'GRIDCODE_150.0', 'ALUM_SAT',
       'GRIDCODE_50.0', 'GRIDCODE_151.0', 'ELEC_COND', 'GRIDCODE_110.0',
       'fire_count'],
      dtype='object')


In [3]:

print("\n=============== FIRE_COUNT DISTRIBUTION ================")
print(merged_df['fire_count'].value_counts().sort_index())
print("==========================================================")



=============== FIRE_COUNT DISTRIBUTION ================
fire_count
0.0    790756
1.0     61197
Name: count, dtype: int64


In [5]:
print(merged_df.head())
print(merged_df.shape)
merged_df

   spring_prec  autumn_prec  summer_prec  winter_prec  autumn_tmax  elevation  \
0     1.891892     1.866102     1.216560    -0.152672    -1.214286   0.942090   
1     1.891892     1.866102     1.216560    -0.152672    -1.214286   0.942090   
2     1.891892     1.866102     1.216560    -0.152672    -1.214286   0.942090   
3    -0.430502     0.232203     0.764331    -0.233779    -0.642857   0.990113   
4    -0.430502     0.232203     0.764331    -0.233779    -0.642857   0.990113   

   autumn_tmin  spring_tmax  summer_tmax  winter_tmin  ...  CEC_SOIL  \
0      -1.2500    -0.666667    -0.666667       -1.250  ... -0.666667   
1      -1.2500    -0.666667    -0.666667       -1.250  ... -3.333333   
2      -1.2500    -0.666667    -0.666667       -1.250  ... -0.666667   
3      -0.5625    -0.500000    -0.083333       -0.875  ... -0.666667   
4      -0.5625    -0.500000    -0.083333       -0.875  ... -3.333333   

     GYPSUM      BSAT  GRIDCODE_150.0  ALUM_SAT  GRIDCODE_50.0  \
0 -0.210526  0

,spring_prec,autumn_prec,summer_prec,winter_prec,autumn_tmax,elevation,autumn_tmin,spring_tmax,summer_tmax,winter_tmin,...,CEC_SOIL,GYPSUM,BSAT,GRIDCODE_150.0,ALUM_SAT,GRIDCODE_50.0,GRIDCODE_151.0,ELEC_COND,GRIDCODE_110.0,fire_count
0,1.891892,1.866102,1.216560,-0.152672,-1.214286,0.942090,-1.2500,-0.666667,-0.666667,-1.250,...,-0.666667,-0.210526,0.045455,False,0.0,False,True,0.0,False,1.0
1,1.891892,1.866102,1.216560,-0.152672,-1.214286,0.942090,-1.2500,-0.666667,-0.666667,-1.250,...,-3.333333,-0.157895,-1.590909,False,0.0,False,True,0.0,False,1.0
2,1.891892,1.866102,1.216560,-0.152672,-1.214286,0.942090,-1.2500,-0.666667,-0.666667,-1.250,...,-0.666667,1.421053,0.000000,False,0.0,False,True,0.0,False,1.0
3,-0.430502,0.232203,0.764331,-0.233779,-0.642857,0.990113,-0.5625,-0.500000,-0.083333,-0.875,...,-0.666667,-0.210526,0.045455,False,0.0,False,True,0.0,False,0.0
4,-0.430502,0.232203,0.764331,-0.233779,-0.642857,0.990113,-0.5625,-0.500000,-0.083333,-0.875,...,-3.333333,-0.157895,-1.590909,False,0.0,False,True,0.0,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
851948,0.689189,1.337288,-0.127389,3.052481,0.285714,-0.905367,0.8750,-0.416667,-1.166667,1.000,...,1.333333,0.526316,-0.772727,False,0.0,False,False,0.0,False,0.0
851949,0.689189,1.337288,-0.127389,3.052481,0.285714,-0.892655,0.8750,-0.416667,-1.166667,1.000,...,-4.000000,-0.263158,-1.045455,False,6.0,False,False,-1.0,False,0.0
851950,0.689189,1.337288,-0.127389,3.052481,0.285714,-0.892655,0.8750,-0.416667,-1.166667,1.000,...,1.333333,0.526316,-0.772727,False,0.0,False,False,0.0,False,0.0
851951,0.833977,1.230508,-0.057325,2.981870,0.285714,-0.776836,0.7500,-0.333333,-0.916667,0.750,...,-4.000000,-0.263158,-1.045455,False,6.0,False,False,-1.0,False,0.0


In [6]:
print(merged_df.columns)

Index(['spring_prec', 'autumn_prec', 'summer_prec', 'winter_prec',
       'autumn_tmax', 'elevation', 'autumn_tmin', 'spring_tmax', 'summer_tmax',
       'winter_tmin', 'spring_tmin', 'winter_tmax', 'summer_tmin', 'TOTAL_N',
       'ORG_CARBON', 'PH_WATER', 'CEC_EFF', 'TCARBON_EQ', 'SILT',
       'GRIDCODE_130.0', 'SAND', 'CN_RATIO', 'GRIDCODE_30.0', 'REF_BULK',
       'GRIDCODE_20.0', 'CEC_CLAY', 'ESP', 'COARSE', 'BULK', 'GRIDCODE_14.0',
       'CEC_SOIL', 'GYPSUM', 'BSAT', 'GRIDCODE_150.0', 'ALUM_SAT',
       'GRIDCODE_50.0', 'GRIDCODE_151.0', 'ELEC_COND', 'GRIDCODE_110.0',
       'fire_count'],
      dtype='object')


In [6]:
merged_df.to_csv("Cleaned_dataset/grid/MERGED_FINAL.csv", index=False)

X = merged_df.drop(columns=["fire_count"])  # <-- your features
y = merged_df["fire_count"]    # <-- your target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y     # <-- preserves the ratio of classes
)

print("Train class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Train class distribution:
fire_count
0.0    0.928168
1.0    0.071832
Name: proportion, dtype: float64

Test class distribution:
fire_count
0.0    0.928171
1.0    0.071829
Name: proportion, dtype: float64


In [ ]:
def balance(train_df, target_col="fire", minority_target=1, desired_minority_prop=0.35, random_state=42, verbose=True):
    
    
    if not (0.0 < desired_minority_prop < 1.0):
        raise ValueError("desired_minority_prop must be between 0 and 1 (exclusive).")

    X = train_df.drop(columns=[target_col])
    y = train_df[target_col].astype(int)


    counts = y.value_counts().to_dict()
    majority_class = y.mode()[0] if len(counts) > 0 else 0
    minority_class = minority_target
    if minority_class not in counts:
        raise ValueError(f"Minority class {minority_class} not in training labels.")

    n_total = len(y)
    n_minority = int(counts.get(minority_class, 0))
    n_majority = int(counts.get(majority_class, 0))

    if verbose:
        print(f"Training size: {n_total} (minority={n_minority}, majority={n_majority})")
        print(f"Desired minority proportion: {desired_minority_prop:.2f}")

    # Neamiss undersampling
    
    approx_final_total = int(max(len(y), np.ceil(n_minority / desired_minority_prop)))


    approx_final_total = max(len(y), int(np.ceil(n_minority / desired_minority_prop)))

    max_majority_after_undersample = max(0, int(approx_final_total * (1 - desired_minority_prop)))

    max_majority_after_undersample = min(max_majority_after_undersample, n_majority)

    if max_majority_after_undersample < n_majority:

        if verbose:
            print(
                f"Undersampling majority from {n_majority} -> {max_majority_after_undersample} (fast reduction)."
            )
        rus = NearMiss(
            sampling_strategy={majority_class: max_majority_after_undersample},
            n_jobs=-1,
        )
        X_res, y_res = rus.fit_resample(X, y)
    else:
        if verbose:
            print("No undersampling applied (majority not reduced).")
        X_res, y_res = X.copy(), y.copy()


    counts_res = pd.Series(y_res).value_counts().to_dict()
    n_minority_res = counts_res.get(minority_class, 0)
    n_majority_res = counts_res.get(majority_class, 0)
    if verbose:
        print(
            f"After undersampling: minority={n_minority_res}, majority={n_majority_res}"
        )

    # SMOTE

    final_total = int(np.ceil(n_minority_res / desired_minority_prop))
    final_total = max(final_total, n_minority_res + n_majority_res)
    m_final = int(np.ceil(desired_minority_prop * final_total))
    m_final = max(m_final, n_minority_res)
    if n_minority_res == 0:
        raise ValueError(
            "No minority samples present after undersampling; SMOTE cannot proceed."
        )
    smote_ratio = m_final / n_minority_res
    sampling_strategy = {minority_class: int(m_final)}

    if verbose:
        print(
            f"SMOTE will generate minority to reach {m_final} samples (ratio={smote_ratio:.2f})."
        )

    smote = SMOTE(sampling_strategy=sampling_strategy, random_state=random_state)
    X_bal, y_bal = smote.fit_resample(X_res, y_res)

    balanced_df = pd.concat(
        [
            pd.DataFrame(X_bal, columns=X.columns).reset_index(drop=True),
            pd.Series(y_bal, name=target_col).reset_index(drop=True),
        ],

        axis=1,
    )

    if verbose:
        final_counts = balanced_df[target_col].value_counts().to_dict()
        print(f"Final balanced sizes: {final_counts} | total={len(balanced_df)}")


    return balanced_df

In [7]:
 
X_train_scaled_df = pd.DataFrame(X_train, columns=X_train.columns)
train_df = pd.concat([X_train_scaled_df.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)

print(train_df.shape)
print(X_test.shape)

balanced_train_df = balance(train_df=train_df, target_col="fire_count" )

X_train_bal = balanced_train_df.drop(columns=["fire_count"])
y_train_bal = balanced_train_df["fire_count"]

print("Train class distribution:")
print(y_train_bal.value_counts(normalize=True))

# Create folder if it doesn't exist
save_dir = "Cleaned_dataset/Balanced_data"
os.makedirs(save_dir, exist_ok=True)

# Save balanced training data
X_train_bal.to_csv(f"{save_dir}/X_train_bal.csv", index=False)
y_train_bal.to_csv(f"{save_dir}/y_train_bal.csv", index=False)

# Save test set (unchanged)
X_test.to_csv(f"{save_dir}/X_test.csv", index=False)
y_test.to_csv(f"{save_dir}/y_test.csv", index=False)

print("All datasets saved inside the 'balanced' folder.")


(681562, 40)
(170391, 39)
Training size: 681562 (minority=48958, majority=632604)
Desired minority proportion: 0.35
Undersampling majority from 632604 -> 443015 (fast reduction).
After undersampling: minority=48958, majority=443015
SMOTE will generate minority to reach 172191 samples (ratio=3.52).
Final balanced sizes: {0: 443015, 1: 172191} | total=615206
Train class distribution:
fire_count
0    0.720108
1    0.279892
Name: proportion, dtype: float64
All datasets saved inside the 'balanced' folder.
